<a href="https://colab.research.google.com/github/JouichatKhadija/.github.io/blob/main/Tp4RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
TP4 - Graph-RAG with Neo4j
Topic: Blockchain, Smart Contracts, Transaction Security
"""

# ===================== IMPORTS =====================
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from neo4j import GraphDatabase

print("="*60)
print("TP4 - Graph-RAG with Neo4j")
print("Topic: Blockchain & Smart Contracts Security")
print("="*60)

# ===================== NEO4J CONNECTION =====================
print("\n[1] Connecting to Neo4j...")

# CONNECTION PARAMETERS
NEO4J_URI = "bolt://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password"  # My Password

try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

    # Test connection
    with driver.session() as session:
        result = session.run("RETURN 'Connected to Neo4j!' as message")
        for record in result:
            print(f"    ✓ {record['message']}")
    print("    ✓ Neo4j connection established!")

except Exception as e:
    print(f"    ✗ Connection failed: {e}")
    import sys; sys.exit()

# ===================== BLOCKCHAIN CORPUS =====================
print("\n[2] Loading Blockchain corpus...")

corpus = [
    "Blockchain is a distributed ledger technology that enables secure transactions.",
    "Smart contracts are self-executing contracts with code on blockchain.",
    "Smart contracts run on Ethereum and automatically enforce agreements.",
    "Transaction security uses cryptographic hashing and consensus mechanisms.",
    "Proof of Work (PoW) is used by Bitcoin to validate transactions.",
    "Proof of Stake (PoS) is an alternative to Proof of Work.",
    "Cryptographic hash functions like SHA-256 ensure transaction integrity.",
    "Digital signatures authenticate transactions on the blockchain.",
    "Ethereum supports smart contracts and decentralized applications.",
    "Immutable ledger means transactions cannot be altered once recorded.",
    "Decentralization eliminates single points of failure.",
    "Consensus mechanisms ensure nodes agree on transaction validity.",
    "Smart contract vulnerabilities can lead to security breaches.",
    "Formal verification proves correctness of smart contract code.",
    "Multi-signature wallets require multiple approvals for transactions."
]

print(f"    Corpus loaded: {len(corpus)} sentences")

# ===================== CHUNKING =====================
print("\n[3] Chunking text...")

def chunk_text(texts, chunk_size=20, overlap=5):
    chunks = []
    for text in texts:
        words = text.split()
        i = 0
        while i < len(words):
            chunk = words[i:i+chunk_size]
            if chunk:
                chunks.append(" ".join(chunk))
            i += chunk_size - overlap
    return chunks

chunks = chunk_text(corpus, chunk_size=20, overlap=5)
print(f"    Created {len(chunks)} chunks")

for i, chunk in enumerate(chunks[:3], 1):
    print(f"    Chunk {i}: {chunk[:70]}...")

# ===================== EMBEDDINGS =====================
print("\n[4] Creating embeddings with SentenceTransformer...")

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks)
embeddings = np.array(embeddings)
print(f"    Embeddings shape: {embeddings.shape}")

# ===================== FAISS INDEX =====================
print("\n[5] Creating FAISS index...")

dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embeddings)
print(f"    FAISS index created with {faiss_index.ntotal} vectors")

# ===================== QUERY =====================
print("\n[6] Processing user query...")

user_query = "smart contract security"
print(f"    Query: '{user_query}'")

query_vector = model.encode([user_query])
query_vector = np.array(query_vector)

k = 3
distances, indices_chunks = faiss_index.search(query_vector, k)
retrieved_chunks = [chunks[i] for i in indices_chunks[0]]

print(f"\n[7] FAISS Retrieval Results (top {k} chunks):")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"    {i}. {chunk[:90]}...")
    print(f"       Distance: {distances[0][i-1]:.4f}")

# ===================== ENTITIES & RELATIONS =====================
print("\n[8] Creating Knowledge Graph in Neo4j...")

entities = [
    "Blockchain", "Smart Contract", "Ethereum", "Bitcoin",
    "Transaction Security", "Cryptographic Hash", "Digital Signature",
    "Proof of Work", "Proof of Stake", "Consensus Mechanism", "SHA-256",
    "Immutable Ledger", "Decentralization", "Smart Contract Vulnerability",
    "Formal Verification", "Multi-signature Wallet"
]

relations = [
    {"source": "Smart Contract", "target": "Blockchain", "relation": "RUNS_ON"},
    {"source": "Ethereum", "target": "Blockchain", "relation": "IS_TYPE_OF"},
    {"source": "Bitcoin", "target": "Blockchain", "relation": "IS_TYPE_OF"},
    {"source": "Proof of Work", "target": "Consensus Mechanism", "relation": "IS_TYPE_OF"},
    {"source": "Proof of Stake", "target": "Consensus Mechanism", "relation": "IS_TYPE_OF"},
    {"source": "Cryptographic Hash", "target": "Transaction Security", "relation": "ENSURES"},
    {"source": "SHA-256", "target": "Cryptographic Hash", "relation": "IS_TYPE_OF"},
    {"source": "Digital Signature", "target": "Transaction Security", "relation": "PROVIDES"},
    {"source": "Immutable Ledger", "target": "Transaction Security", "relation": "ENSURES"},
    {"source": "Smart Contract Vulnerability", "target": "Security Breach", "relation": "CAUSES"},
    {"source": "Formal Verification", "target": "Smart Contract", "relation": "SECURES"},
    {"source": "Consensus Mechanism", "target": "Transaction", "relation": "VALIDATES"},
    {"source": "Proof of Work", "target": "Bitcoin", "relation": "USED_BY"},
    {"source": "Decentralization", "target": "Transaction Security", "relation": "IMPROVES"},
    {"source": "Multi-signature Wallet", "target": "Transaction Security", "relation": "ENHANCES"}
]

def create_knowledge_graph(tx, entities, relations):
    # Clear existing data
    tx.run("MATCH (n) DETACH DELETE n")
    print("    Cleared existing data")

    # Create nodes
    for entity in entities:
        tx.run("MERGE (n:Entity {name: $name})", name=entity)
    print(f"    Created {len(entities)} nodes")

    # Create relationships
    for rel in relations:
        tx.run("""
            MATCH (a:Entity {name: $source})
            MATCH (b:Entity {name: $target})
            MERGE (a)-[:RELATION {type: $rel}]->(b)
        """, source=rel["source"], target=rel["target"], rel=rel["relation"])
    print(f"    Created {len(relations)} relationships")

with driver.session() as session:
    session.execute_write(create_knowledge_graph, entities, relations)
    print("    ✓ Knowledge Graph created in Neo4j!")

# ===================== QUERY NEO4J =====================
print(f"\n[9] Querying Neo4j for '{user_query}'...")

def query_neo4j(driver, term):
    cypher = """
        MATCH (n:Entity)-[r:RELATION]->(m:Entity)
        WHERE n.name CONTAINS $term OR m.name CONTAINS $term
        RETURN n.name as source, r.type as relation, m.name as target
        LIMIT 15
    """
    with driver.session() as session:
        result = session.run(cypher, term=term)
        return [{"source": r["source"], "relation": r["relation"], "target": r["target"]} for r in result]

results = query_neo4j(driver, "Smart Contract")

print(f"\n[10] Neo4j Query Results:")
if results:
    print(f"    Found {len(results)} relationships:")
    for rel in results:
        print(f"    • {rel['source']} --[{rel['relation']}]--> {rel['target']}")
else:
    print("    No relationships found")

# ===================== QUERY FOR SPECIFIC ENTITY =====================
print("\n[11] Querying for 'Smart Contract' relations:")

def get_entity_relations(driver, entity):
    cypher = """
        MATCH (n:Entity {name: $entity})-[r:RELATION]->(m:Entity)
        RETURN n.name as source, r.type as relation, m.name as target
        UNION
        MATCH (n:Entity)-[r:RELATION]->(m:Entity {name: $entity})
        RETURN n.name as source, r.type as relation, m.name as target
    """
    with driver.session() as session:
        result = session.run(cypher, entity=entity)
        return [{"source": r["source"], "relation": r["relation"], "target": r["target"]} for r in result]

sc_results = get_entity_relations(driver, "Smart Contract")
for rel in sc_results:
    print(f"    • {rel['source']} --[{rel['relation']}]--> {rel['target']}")

# ===================== QUERY FOR RETRIEVED CHUNKS =====================
print("\n[12] Finding Neo4j relationships for retrieved chunks keywords...")

keywords = ["Blockchain", "Smart Contract", "Transaction Security", "Ethereum"]

for keyword in keywords:
    relations_found = get_entity_relations(driver, keyword)
    if relations_found:
        print(f"\n    Relations for '{keyword}':")
        for rel in relations_found[:3]:
            print(f"      → {rel['source']} --[{rel['relation']}]--> {rel['target']}")

# ===================== CLOSE CONNECTION =====================
driver.close()
print("\n[13] Neo4j connection closed.")

# ===================== FINAL SUMMARY =====================
print("\n" + "="*60)
print("TP4 - Graph-RAG with Neo4j COMPLETED!")
print("="*60)
print("\n📊 EXECUTION SUMMARY:")
print(f"   • Blockchain corpus: {len(corpus)} sentences")
print(f"   • Text chunks created: {len(chunks)}")
print(f"   • Embeddings dimension: {embeddings.shape[1]}")
print(f"   • FAISS index size: {faiss_index.ntotal} vectors")
print(f"   • User query: '{user_query}'")
print(f"   • FAISS retrieved: {len(retrieved_chunks)} relevant chunks")
print(f"   • Neo4j entities: {len(entities)}")
print(f"   • Neo4j relations: {len(relations)}")
print(f"   • Neo4j query results: {len(results)}")
print("\n✅ TP4 COMPLETED SUCCESSFULLY!")
print("\n💡 TO VIEW THE GRAPH IN NEO4J BROWSER:")
print("   1. In Neo4j Desktop, click 'Open Browser'")
print("   2. Run: MATCH (n) RETURN n LIMIT 25")
print("   3. Run: MATCH (n)-[r]->(m) RETURN n,r,m LIMIT 30")